# SmartLearn — recommandations TF-IDF et similarité cosinus

Notebook pédagogique reproduisant le calcul de `mla-service/main.py`.

**Données :** 15 formations et 4 profils entièrement synthétiques, inclus dans ce notebook et fournis dans `dataset_smartlearn.json`. Ils ne sont pas extraits de MySQL.

**Exécution :** ouvrir dans Jupyter ou Google Colab, puis exécuter les cellules dans l’ordre. Si nécessaire, décommenter la commande d’installation. Aucun serveur ni connexion à MySQL n’est nécessaire.

Il s’agit de recommandation basée sur le contenu. Il n’y a pas d’entraînement supervisé ni de prédiction de réussite.

In [1]:
# Si scikit-learn est absent, décommenter puis exécuter :
# %pip install scikit-learn
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Charger le dataset
Le dataset est intégré pour que le notebook fonctionne seul. Le champ `nom` sert uniquement à identifier les profils dans les exemples ; il ne participe pas au calcul.

In [2]:
dataset = json.loads('{"description": "Dataset pédagogique synthétique, sans données personnelles ni labels de pertinence réels.", "formations": [{"id": 1, "titre": "Java pour débutants", "description": "Introduction à Java et à la programmation orientée objet.", "niveau": "Débutant", "motsCles": "java programmation classes objets", "categorie": "Développement Web"}, {"id": 2, "titre": "Spring Boot Avancé", "description": "Développement API REST avec Java Spring Boot et SQL pour le backend.", "niveau": "Avancé", "motsCles": "java spring boot backend api rest sql", "categorie": "Développement Web"}, {"id": 3, "titre": "Angular Fundamentals", "description": "Créer des interfaces web avec Angular et TypeScript.", "niveau": "Intermédiaire", "motsCles": "angular typescript frontend composants", "categorie": "Développement Web"}, {"id": 4, "titre": "Python pour débutants", "description": "Apprendre les bases de Python et écrire des scripts.", "niveau": "Débutant", "motsCles": "python variables boucles fonctions", "categorie": "Programmation"}, {"id": 5, "titre": "Analyse de données avec Python", "description": "Analyser des données avec pandas et réaliser des visualisations.", "niveau": "Intermédiaire", "motsCles": "python pandas analyse données statistiques", "categorie": "Data Science"}, {"id": 6, "titre": "Machine Learning avec scikit-learn", "description": "Construire et évaluer des modèles de classification et régression.", "niveau": "Intermédiaire", "motsCles": "python machine learning scikit learn modèles données", "categorie": "Data Science"}, {"id": 7, "titre": "SQL et bases de données", "description": "Écrire des requêtes SQL et concevoir des bases de données relationnelles.", "niveau": "Débutant", "motsCles": "sql mysql requêtes jointures bases données", "categorie": "Bases de données"}, {"id": 8, "titre": "Docker et CI/CD", "description": "Déployer des applications avec Docker et automatiser les pipelines CI/CD.", "niveau": "Intermédiaire", "motsCles": "docker devops déploiement pipeline ci cd", "categorie": "Cloud / DevOps"}, {"id": 9, "titre": "Kubernetes en pratique", "description": "Orchestrer des conteneurs et gérer des déploiements dans le cloud.", "niveau": "Avancé", "motsCles": "kubernetes docker cloud devops orchestration", "categorie": "Cloud / DevOps"}, {"id": 10, "titre": "Sécurité des applications web", "description": "Sécuriser les API et comprendre les vulnérabilités web.", "niveau": "Intermédiaire", "motsCles": "sécurité web api authentification vulnérabilités", "categorie": "Cybersécurité"}, {"id": 11, "titre": "Management équipe", "description": "Développer le leadership et la communication pour gérer une équipe.", "niveau": "Débutant", "motsCles": "management leadership communication équipe", "categorie": "Management"}, {"id": 12, "titre": "Gestion de projet agile", "description": "Organiser les projets avec Scrum et les méthodes agiles.", "niveau": "Intermédiaire", "motsCles": "gestion projet agile scrum équipe", "categorie": "Management"}, {"id": 13, "titre": "Design UX/UI", "description": "Concevoir des interfaces et améliorer expérience utilisateur.", "niveau": "Débutant", "motsCles": "design ux ui interfaces utilisateur figma", "categorie": "Design"}, {"id": 14, "titre": "React et JavaScript", "description": "Développer des interfaces frontend avec React et JavaScript.", "niveau": "Intermédiaire", "motsCles": "react javascript frontend web composants", "categorie": "Développement Web"}, {"id": 15, "titre": "Tests Java avec JUnit", "description": "Tester les applications Java et Spring Boot avec JUnit et Mockito.", "niveau": "Intermédiaire", "motsCles": "java spring boot junit mockito tests backend", "categorie": "Développement Web"}], "profils": [{"nom": "Profil backend fictif", "niveau": "Intermédiaire", "objectif": "Développeur Backend", "domaineInteret": "Développement Web", "competences": ["Java", "Spring", "SQL"]}, {"nom": "Profil data fictif", "niveau": "Intermédiaire", "objectif": "Analyse de données avec Python", "domaineInteret": "Data Science", "competences": ["Python", "Pandas", "Statistiques"]}, {"nom": "Profil DevOps fictif", "niveau": "Intermédiaire", "objectif": "Déploiement cloud et DevOps", "domaineInteret": "Cloud / DevOps", "competences": ["Docker", "Kubernetes", "CI/CD"]}, {"nom": "Profil management fictif", "niveau": "Débutant", "objectif": "Management équipe et leadership", "domaineInteret": "Management", "competences": ["Communication", "Gestion projet"]}]}')
formations = dataset["formations"]
profils = dataset["profils"]
print(f"{len(formations)} formations, {len(profils)} profils")
for f in formations:
    print(f"{f['id']:2} | {f['titre']} | {f['categorie']}")

15 formations, 4 profils
 1 | Java pour débutants | Développement Web
 2 | Spring Boot Avancé | Développement Web
 3 | Angular Fundamentals | Développement Web
 4 | Python pour débutants | Programmation
 5 | Analyse de données avec Python | Data Science
 6 | Machine Learning avec scikit-learn | Data Science
 7 | SQL et bases de données | Bases de données
 8 | Docker et CI/CD | Cloud / DevOps
 9 | Kubernetes en pratique | Cloud / DevOps
10 | Sécurité des applications web | Cybersécurité
11 | Management équipe | Management
12 | Gestion de projet agile | Management
13 | Design UX/UI | Design
14 | React et JavaScript | Développement Web
15 | Tests Java avec JUnit | Développement Web


## 2. Construire les textes
Même ordre de champs que le service Python. Les textes passent en minuscules. Les noms des compétences sont utilisés, mais pas leurs pourcentages de maîtrise. Le niveau est traité comme un mot, sans contrainte pédagogique.

In [3]:
def texte_profil(profil):
    competences = " ".join(profil.get("competences") or [])
    return f"{profil.get('objectif') or ''} {profil.get('domaineInteret') or ''} {competences} {profil.get('niveau') or ''}".strip().lower()

def texte_formation(f):
    return f"{f['titre']} {f.get('description') or ''} {f.get('motsCles') or ''} {f.get('categorie') or ''} {f.get('niveau') or ''}".strip().lower()

profil = profils[0]
corpus = [texte_profil(profil)] + [texte_formation(f) for f in formations]
print("PROFIL :", corpus[0])
print("FORMATION :", corpus[2])

PROFIL : développeur backend développement web java spring sql intermédiaire
FORMATION : spring boot avancé développement api rest avec java spring boot et sql pour le backend. java spring boot backend api rest sql développement web avancé


## 3. Transformer les textes avec TF-IDF

`ngram_range=(1, 2)` conserve les mots et les paires de mots, par exemple `spring` et `spring boot`.

Avec les paramètres par défaut utilisés ici, le poids est basé sur la fréquence du terme multipliée par `IDF = log((1 + N) / (1 + df)) + 1`, puis le vecteur est normalisé en norme L2. `N` est le nombre de documents (profil inclus) et `df` le nombre de documents contenant le terme.

Comme dans le projet, `fit_transform` est effectué sur le profil et le catalogue à chaque demande. Changer le catalogue peut donc changer les scores. Aucun traitement spécifique des synonymes ni liste française de mots vides n’est ajouté.

In [4]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
matrice = vectorizer.fit_transform(corpus)
print("Dimensions (documents, termes) :", matrice.shape)
termes = vectorizer.get_feature_names_out()
poids = matrice[0].toarray()[0]
for i in poids.argsort()[::-1][:12]:
    if poids[i] > 0:
        print(f"{termes[i]:25} {poids[i]:.4f}")

Dimensions (documents, termes) : (16, 374)
sql intermédiaire         0.3168
spring sql                0.3168
développeur backend       0.3168
développeur               0.3168
web java                  0.3168
backend développement     0.2759
spring                    0.2469
java spring               0.2469
sql                       0.2469
backend                   0.2469
java                      0.2244
développement web         0.1904


## 4. Calculer la similarité cosinus

`cosinus(u, v) = (u · v) / (||u|| × ||v||)`

On compare le premier vecteur (profil) à chacun des suivants (formations). Avec ces vecteurs TF-IDF non négatifs, les scores sont compris entre 0 et 1. Un affichage de 19 % signifie une similarité de 0,19, pas une probabilité de réussite ni une précision du système.

In [5]:
similarites = cosine_similarity(matrice[0:1], matrice[1:])[0]
for f, score in zip(formations, similarites):
    print(f"{f['titre']:38} {score:.4f}")

Java pour débutants                    0.1459
Spring Boot Avancé                     0.3162
Angular Fundamentals                   0.0862
Python pour débutants                  0.0000
Analyse de données avec Python         0.0111
Machine Learning avec scikit-learn     0.0102
SQL et bases de données                0.0664
Docker et CI/CD                        0.0092
Kubernetes en pratique                 0.0000
Sécurité des applications web          0.0600
Management équipe                      0.0000
Gestion de projet agile                0.0112
Design UX/UI                           0.0000
React et JavaScript                    0.0738
Tests Java avec JUnit                  0.2499


## 5. Fonction complète identique au calcul du projet
Le score est arrondi à deux décimales **avant le tri**, comme dans `main.py`. Les égalités conservent l’ordre du catalogue. Le service retourne toutes les formations ; le top 5 ci-dessous est seulement un choix d’affichage.

In [6]:
def recommander(profil, formations):
    if not formations:
        return []
    corpus = [texte_profil(profil)] + [texte_formation(f) for f in formations]
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    if not any(vectorizer.build_analyzer()(texte) for texte in corpus):
        return [{"formationId": f["id"], "titre": f["titre"], "score": 0.0} for f in formations]
    matrice = vectorizer.fit_transform(corpus)
    scores = cosine_similarity(matrice[0:1], matrice[1:])[0]
    resultats = [{"formationId": f["id"], "titre": f["titre"], "score": round(float(s), 2)}
                 for f, s in zip(formations, scores)]
    resultats.sort(key=lambda r: r["score"], reverse=True)
    return resultats

for p in profils:
    print("\n" + p["nom"])
    for r in recommander(p, formations)[:5]:
        print(f"  {r['score']:.0%} | {r['titre']}")


Profil backend fictif
  32% | Spring Boot Avancé
  25% | Tests Java avec JUnit
  15% | Java pour débutants
  9% | Angular Fundamentals
  7% | SQL et bases de données

Profil data fictif
  59% | Analyse de données avec Python
  16% | Python pour débutants
  15% | Machine Learning avec scikit-learn
  15% | SQL et bases de données
  3% | Gestion de projet agile

Profil DevOps fictif
  44% | Docker et CI/CD
  34% | Kubernetes en pratique
  2% | React et JavaScript
  1% | Angular Fundamentals
  1% | Analyse de données avec Python

Profil management fictif
  41% | Management équipe
  23% | Gestion de projet agile
  2% | Java pour débutants
  2% | Python pour débutants
  2% | SQL et bases de données


## 6. Tester ton propre profil
Modifie les champs puis réexécute la cellule. Des scores faibles sont possibles même pour une formation utile : ce modèle compare les mots employés.

In [7]:
mon_profil = {
    "objectif": "Développeur Backend Java",
    "domaineInteret": "Développement Web",
    "niveau": "Intermédiaire",
    "competences": ["Java", "Spring", "SQL"]
}
for r in recommander(mon_profil, formations)[:5]:
    print(f"{r['score']:.0%} | {r['titre']}")

35% | Spring Boot Avancé
25% | Tests Java avec JUnit
21% | Java pour débutants
8% | Angular Fundamentals
7% | React et JavaScript


## 7. Vérifier le comportement
Ces assertions vérifient le fonctionnement, pas la qualité pédagogique des recommandations. Sans jugements de pertinence indépendants, on ne peut pas annoncer une accuracy ou une précision réelle.

In [8]:
resultats = recommander(profils[0], formations)
assert len(resultats) == len(formations)
assert all(0 <= r["score"] <= 1 for r in resultats)
assert resultats == sorted(resultats, key=lambda r: r["score"], reverse=True)
assert recommander(profils[0], []) == []
assert all(r["score"] == 0 for r in recommander({}, formations))
assert recommander({}, [{"id": 99, "titre": ""}])[0]["score"] == 0
print("Vérifications fonctionnelles réussies.")

Vérifications fonctionnelles réussies.


## 8. Limites et lien avec SmartLearn

- Le notebook reproduit le moteur Python, pas la méthode Java de secours `MOTS_CLES`.
- FastAPI expose ce calcul via `POST /recommend` ; Spring Boot fournit les données de MySQL et Angular affiche les résultats.
- Les scores reposent sur la proximité textuelle, sans compréhension des synonymes, des prérequis ou de la progression.
- Les notes, clics et formations terminées ne sont pas utilisés.
- Pour évaluer la pertinence, faire annoter des paires profil/formation par des évaluateurs, puis mesurer par exemple Precision@5 ou NDCG@5 sur des données réservées à l’évaluation. Le dataset synthétique fourni sert à expliquer et tester le code, pas à prouver sa performance réelle.
